# 精确 f696 的 SOXL 对现金配置意图排序

只读冻结 Qlib 标签、提案及完整账户窗口，检验原有配置排序与未来成对 H5 毛价格优势的关系；没有拟合、动作规则或新账户。

In [1]:
import os
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

root=Path.cwd();card=root/'research/stage1_f696_pairwise_intent_rank_20261024'
plan_path=card/'plan.json';plan=json.loads(plan_path.read_text());src=plan['sources']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_market_GETs','new_model_fits','new_native_accounts','new_bootstraps','new_Recorders'))
assert sha(root/src['geometry_plan'])==src['geometry_plan_sha256']
a=root/src['labels_card'];f=root/src['f696_card'];b=root/src['account_card']
for file,key in [(a/'summary.json','labels_summary_sha256'),(f/'summary.json','f696_summary_sha256'),(b/'summary.json','account_summary_sha256'),(root/src['regime_plan'],'regime_plan_sha256')]:assert sha(file)==src[key],key
sa=json.loads((a/'summary.json').read_text());sf=json.loads((f/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sa['nas_evidence']['sha256']==src['labels_NAS_sha256'] and sa['nas_evidence']['size_bytes']==src['labels_NAS_size_bytes'] and sa['nas_readback']['verified']
assert sf['nas_evidence']['sha256']==src['f696_NAS_sha256'] and sf['nas_evidence']['size_bytes']==src['f696_NAS_size_bytes']
assert sb['nas_readback']['verified'] and sb['nas_readback']['sha256']==src['account_NAS_sha256']
aa=a/'qlib_recorder'/src['labels_experiment_id']/src['labels_recorder_id']/'artifacts/inputs/dataset'
fa=f/'qlib_recorder'/src['f696_experiment_id']/src['f696_recorder_id']/'artifacts/inputs/frame'
ba=b/'qlib_recorder'/src['account_experiment_id']/src['account_recorder_id']/'artifacts/analysis/rolling126'
assert sha(aa)==src['labels_dataset_sha256'] and sha(fa)==src['f696_frame_sha256'] and sha(ba)==src['account_rolling_sha256']
assert sha(root/src['qlib_provider']/'qlib_store_manifest.json')==src['provider_manifest_sha256']
qlib.init(provider_uri=str((root/src['qlib_provider']).resolve()),region='us',expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str((a/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=src['labels_experiment_id'],recorder_id=src['labels_recorder_id']);dataset=rec.load_object('inputs/dataset')
with R.uri_context(str((f/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=src['f696_experiment_id'],recorder_id=src['f696_recorder_id']);frame=rec.load_object('inputs/frame').sort_values('trade_date').reset_index(drop=True)
with R.uri_context(str((b/'qlib_recorder').resolve())):
 rec=R.get_recorder(experiment_id=src['account_experiment_id'],recorder_id=src['account_recorder_id']);rolling=rec.load_object('analysis/rolling126')
train=dataset.prepare('train',col_set=['label','meta']);held=dataset.prepare('held',col_set=['label','meta'])
train=train.loc[train[('meta','train_mature')].astype(bool)];held=held.loc[held[('meta','held_mature')].astype(bool)]
assert (len(train),len(held),len(frame),len(rolling))==(3014,940,944,819)
held_dates=pd.DatetimeIndex(held.index.get_level_values('datetime'));fdates=pd.DatetimeIndex(frame.signal_date)
assert held_dates.is_unique and fdates.is_unique and held_dates.equals(fdates[:940])
train_margin=train[('label','SOXL_relative_H5')].to_numpy(float)-train[('label','cash_advantage_H5')].to_numpy(float)
held_margin=held[('label','SOXL_relative_H5')].to_numpy(float)-held[('label','cash_advantage_H5')].to_numpy(float)
assert np.isfinite(train_margin).all() and np.isfinite(held_margin).all()
frame['pair_margin']=np.r_[held_margin,[np.nan]*4]
frame['intent_score']=frame.soxl_weight.to_numpy(float)-frame.cash_weight.to_numpy(float)
assert np.isfinite(frame.intent_score).all() and frame.pair_margin.notna().sum()==940
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'))
assert fdates.isin(calendar).all()

def rank(g):
 known=g.loc[np.isfinite(g.pair_margin)].copy();binary=known.loc[known.pair_margin.abs()>1e-12].copy()
 y=(binary.pair_margin>0).to_numpy(int);score=binary.intent_score.to_numpy(float)
 auc=float(roc_auc_score(y,score)) if len(set(y))==2 else None
 sp=float(spearmanr(known.intent_score.to_numpy(float),known.pair_margin.to_numpy(float)).statistic) if len(known)>2 and known.intent_score.nunique()>1 else None
 median=float(np.median(score)) if len(score) else None
 low=y[score<median];high=y[score>median]
 return {'sessions':len(g),'mature':len(known),'unknown':len(g)-len(known),'positive':int(y.sum()),'negative':int(len(y)-y.sum()),'ties':int(len(known)-len(binary)),
         'auc':auc,'spearman_continuous':sp,'mean_intent_score':float(g.intent_score.mean()),
         'median_split':{'score_median':median,'low_rows':len(low),'low_positive_fraction':float(low.mean()) if len(low) else None,'high_rows':len(high),'high_positive_fraction':float(high.mean()) if len(high) else None}}
held_full=rank(frame);annual=[];quarter=[]
for year,g in frame.groupby(pd.to_datetime(frame.trade_date).dt.year,sort=True):annual.append({'year':int(year),'partial_endpoint':year in (2023,2026),**rank(g)})
for (year,q),g in frame.groupby([pd.to_datetime(frame.trade_date).dt.year,pd.to_datetime(frame.trade_date).dt.quarter],sort=True):quarter.append({'year':int(year),'quarter':int(q),**rank(g)})
halves=[{'start':'2023-01-03','end':'2024-12-31',**rank(frame.iloc[:502])},{'start':'2025-01-02','end':'2026-10-07',**rank(frame.iloc[502:])}]
assert len(annual)==4 and len(quarter)==16 and held_full['mature']==940 and halves[0]['mature']==502 and halves[1]['mature']==438
windows=[]
for i in range(819):
 rw=rolling.iloc[i];g=frame.iloc[i:i+126]
 assert pd.Timestamp(g.trade_date.iloc[0])==pd.Timestamp(rw.first_trade)
 signal=pd.Timestamp(g.signal_date.iloc[0]);pos=calendar.get_loc(signal);assert pos>=20
 # Prior-state mapping is frozen in the geometry card; use its exact source Qlib daily price feature.
 windows.append({'year':int(pd.Timestamp(rw.first_trade).year),'outcome':'weak' if rw.relative_log_excess<0 else 'good' if rw.relative_log_excess>0 else 'tie',**rank(g),'signal_date':str(signal.date())})
assert pd.Series([w['outcome'] for w in windows]).value_counts().to_dict()=={'weak':564,'good':255}
# Exact frozen start-state labels require the same Qlib SOXX close query as the geometry card.
close=D.features(['SOXX'],['$close'],start_time='2011-01-03',end_time='2026-10-09',freq='day')['$close'].droplevel('instrument').sort_index()
assert close.index.equals(calendar)
for row in windows:
 pos=calendar.get_loc(pd.Timestamp(row['signal_date']));r5=float(close.iloc[pos]/close.iloc[pos-5]-1);r20=float(close.iloc[pos]/close.iloc[pos-20]-1)
 row['start_state']='uptrend' if r5>=0 and r20>=0 else 'rebound' if r5>=0 else 'pullback' if r20>=0 else 'decline'
w=pd.DataFrame(windows)
assert w.mature.min()>=122 and w.mature.max()==126 and w.auc.notna().all()
cells=[]
for (state,year),g in w.groupby(['start_state','year'],sort=True):
 weak=g[g.outcome.eq('weak')];good=g[g.outcome.eq('good')]
 row={'start_state':state,'year':int(year),'weak_windows':len(weak),'good_windows':len(good),'matched':bool(len(weak) and len(good))}
 if row['matched']:row['weak_minus_good_auc']=float(weak.auc.mean()-good.auc.mean())
 cells.append(row)
matched=[x for x in cells if x['matched']];assert len(matched)==12 and all(x['good_windows']==0 for x in cells if x['year']==2023)
weights=np.array([min(x['weak_windows'],x['good_windows']) for x in matched],float)
matched_auc_diff=float(np.average([x['weak_minus_good_auc'] for x in matched],weights=weights))
rolling_summary={name:{'windows':len(g),'mean_auc':float(g.auc.mean()),'fraction_auc_above_half':float((g.auc>0.5).mean()),'auc_p10_p50_p90':[float(g.auc.quantile(q)) for q in (.1,.5,.9)]} for name,g in w.groupby('outcome')}
result={'status':'read_only_f696_pairwise_intent_rank_diagnostic_no_policy','plan_sha256':sha(plan_path),'source_recorder_ids':{k:src[k+'_recorder_id'] for k in ('labels','f696','account')},
        'train_mature_rows':len(train),'train_SOXL_over_cash':int((train_margin>1e-12).sum()),'held_mature_rows':len(held),'held_SOXL_over_cash':held_full['positive'],
        'held_full':held_full,'halves':halves,'annual':annual,'quarter':quarter,'rolling126_windows':len(w),'rolling126_summary':rolling_summary,
        'matched_state_year_cells':len(matched),'matched_weak_minus_good_auc':matched_auc_diff,'cells':cells,
        'new_market_GETs':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Recorders':0,
        'independent_validation':False,'SOXX_goal_achieved':False,
        'limits':['H5 price margin is future and overlapping, with no dividends, risk, Tiered fees or funded-account outcomes in this rank test',
                  'f696 proposal weights are pre-risk intent and 2023-2026 held market history has been reused',
                  'Overlapping weak/good windows and median splits cannot define a tradable switch or infer causation']}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'full_auc':held_full['auc'],'year_auc':[(x['year'],x['auc']) for x in annual],'matched_weak_minus_good_auc':matched_auc_diff,'train_positive':result['train_SOXL_over_cash'],'held_positive':result['held_SOXL_over_cash']})


{'full_auc': 0.46799513276178756, 'year_auc': [(2023, 0.3761194029850746), (2024, 0.4870725604670559), (2025, 0.5007384532760473), (2026, 0.5070195627157652)], 'matched_weak_minus_good_auc': -0.03563099810487807, 'train_positive': 1697, 'held_positive': 539}


## 中文结论

# f696 的 SOXL 对现金配置意图在弱区间反向排序

只读封存的原39项 Qlib H5 标签、精确 f696 09:29 原提案及原0bp Tiered账户819个滚动126日窗口；逐件核对 SHA、NAS 收据/限定回读、3014/940成熟标签、944提案和Qlib日历。固定分数为 `soxl_weight-cash_weight`，未来成对标签为 `SOXL_relative_H5-cash_advantage_H5` 的正号；没有拟合、改符号、挑阈值或新账户。最后4个提案日的标签保持未知。

在940个成熟复用评价日，SOXL未来H5毛价格胜现金539日、负401日。f696原提案分数辨别正号的AUC **0.4680**，连续边际Spearman **-0.0235**；分数中位数以下/以上的未来胜率分别 **61.9%/52.8%**，方向与预定假说相反。2023部分年AUC **0.3761**，完整2024 **0.4871**、完整2025 **0.5007**、2026部分年 **0.5070**；前后两半 **0.4262/0.4986**。16季度均计算并见 `result.json`。

原账户564个未胜SOXX滚动窗口，成对排序平均AUC **0.3671**，255个胜出窗口 **0.4856**。按事前SOXX 5/20日走势状态及窗口首交易年配对的12格，好坏差（弱减好）的加权平均AUC **-0.0356**；2023四格全无同格好窗口。819窗口全部纳入计算，分布及配对格子见 `result.json`；窗口彼此高度重叠，这些差异不是独立统计检验。

这把问题定位到**原配置意图与SOXL/现金相对机会不一致**，尤其2023及账户弱窗口；不能只归咎于后续平滑或开盘风险上限。原提案也并非专为预测此H5标签而训练，故AUC低不能证明原因子完全无信息。下一步应在训练期识别哪个已知因子家族使现金意图压过SOXL，再冻结一种不同于已失败双Ridge头的决策前成对机制与强对照；若信息通过，再检验配置映射与自身风险容量的完整0bp含费账户。历史评价期已复用，毛价格标签不含分红、Tiered费或资金状态；没有SOXX超额或前向通过。决定 `descriptive_f696_pairwise_SOXL_cash_intent_misrank_no_policy`。
